# google_auth_oauthlib smoke test

Smoke test for `google-auth-oauthlib` (oauthlib integration for google-auth).
All examples run offline — OAuth flows are built from an in-memory fake client
config; no token exchange is ever attempted.

In [ ]:
# Example 1: Build an InstalledAppFlow from an in-memory client config (offline).
from google_auth_oauthlib.flow import Flow, InstalledAppFlow

CLIENT_CONFIG = {
    "installed": {
        "client_id": "fake-client-id.apps.googleusercontent.com",
        "client_secret": "fake-secret",
        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
        "token_uri": "https://oauth2.googleapis.com/token",
        "redirect_uris": ["http://localhost"],
    }
}

flow = InstalledAppFlow.from_client_config(CLIENT_CONFIG, scopes=["openid", "email"])
print(f"flow type: {type(flow).__name__}")
print(f"client_id: {flow.client_config['client_id']}")
assert isinstance(flow, Flow)
assert flow.client_config["client_id"].endswith("googleusercontent.com")

In [ ]:
# Example 2: Generate the authorization URL offline (no token exchange).
from urllib.parse import parse_qs, urlparse

from google_auth_oauthlib.flow import InstalledAppFlow

CLIENT_CONFIG = {
    "installed": {
        "client_id": "fake-client-id.apps.googleusercontent.com",
        "client_secret": "fake-secret",
        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
        "token_uri": "https://oauth2.googleapis.com/token",
        "redirect_uris": ["http://localhost"],
    }
}

flow = InstalledAppFlow.from_client_config(CLIENT_CONFIG, scopes=["openid"])
flow.redirect_uri = "http://localhost:8080/"
auth_url, state = flow.authorization_url(prompt="consent")

parsed = urlparse(auth_url)
qs = parse_qs(parsed.query)
print(f"host: {parsed.netloc}  path: {parsed.path}")
print(f"state: {state}")
assert parsed.netloc == "accounts.google.com"
assert qs["client_id"][0] == "fake-client-id.apps.googleusercontent.com"
assert qs["state"][0] == state
assert qs["prompt"][0] == "consent"

In [ ]:
# Example 3: helpers.session_from_client_config builds a requests-oauthlib session.
from google_auth_oauthlib.helpers import session_from_client_config

CLIENT_CONFIG = {
    "installed": {
        "client_id": "fake-client-id.apps.googleusercontent.com",
        "client_secret": "fake-secret",
        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
        "token_uri": "https://oauth2.googleapis.com/token",
        "redirect_uris": ["http://localhost"],
    }
}

session, config = session_from_client_config(CLIENT_CONFIG, scopes=["openid", "email"])
print(f"session type: {type(session).__name__}")
print(f"session scope: {session.scope}")
assert type(session).__name__ == "OAuth2Session"
assert config == CLIENT_CONFIG
print("session_from_client_config round-trips the client config")

In [ ]:
# Example 4: Web-app Flow with an explicit redirect_uri and a caller-provided state.
from google_auth_oauthlib.flow import Flow

WEB_CONFIG = {
    "web": {
        "client_id": "fake-web-client.apps.googleusercontent.com",
        "client_secret": "fake-secret",
        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
        "token_uri": "https://oauth2.googleapis.com/token",
    }
}

flow = Flow.from_client_config(
    WEB_CONFIG, scopes=["openid"], redirect_uri="https://myapp.example/callback"
)
auth_url, state = flow.authorization_url(state="fixed-state-123")
print(f"redirect_uri: {flow.redirect_uri}")
print(f"returned state: {state}")
assert flow.redirect_uri == "https://myapp.example/callback"
assert state == "fixed-state-123"
assert "redirect_uri=https" in auth_url

In [ ]:
# Example 5: The google-oauthlib-tool console script + top-level convenience API.
import importlib.metadata as md
import inspect

import google_auth_oauthlib

dist = md.distribution("google-auth-oauthlib")
eps = [ep for ep in dist.entry_points if ep.group == "console_scripts"]
print(f"console scripts: {[ep.name for ep in eps]}")
assert any(ep.name == "google-oauthlib-tool" for ep in eps)

# get_user_credentials is the documented top-level helper (network only when called).
sig = inspect.signature(google_auth_oauthlib.get_user_credentials)
print(f"get_user_credentials{sig}")
params = list(sig.parameters)
assert "scopes" in params and "client_id" in params and "client_secret" in params
print("public convenience API surface intact")